<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_05_Data_Cleaning_and_Preprocessing/Week_1_Handling_Missing_Data/Day_06_Handling_Duplicates_and_Outliers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 6: Handling Duplicates and Outliers

## Phase 2 | Month 5 | Week 1 | Day 6

**🎯 Goal:** Detect and handle duplicate records and outliers using statistical and rule-based methods.
**⏱️ Study Session:** ~2 hours

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(5)
N = 600
# Safaricom subscriber database with duplicates
df = pd.DataFrame({
    'subscriber_id': list(range(1, N+1)) + [100, 250, 400, 500],
    'msisdn':        ['07' + str(np.random.randint(10000000,99999999)) for _ in range(N)] +
                     ['0712345678','0723456789','0734567890','0745678901'],
    'data_gb':       np.concatenate([np.random.lognormal(1, 0.8, N), [0.001, 500, 0.0, 250]]),  # outliers
    'arpu_kes':      np.concatenate([np.random.lognormal(6.5, 0.7, N), [5, 50000, 3, 48000]]),
})

print(f'Total rows:          {len(df)}')
print(f'Duplicate sub IDs:   {df.duplicated(subset="subscriber_id").sum()}')
print(f'Duplicate MSISDNs:   {df.duplicated(subset="msisdn").sum()}')
print(f'Duplicate full rows: {df.duplicated().sum()}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(5)
N = 600
df = pd.DataFrame({
    'data_gb':  np.concatenate([np.random.lognormal(1, 0.8, N), [0.001, 500, 0.0, 250]]),
    'arpu_kes': np.concatenate([np.random.lognormal(6.5, 0.7, N), [5, 50000, 3, 48000]]),
})

# IQR method for outlier detection
def iqr_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5*iqr, q3 + 1.5*iqr

# Z-score method
def zscore_outliers(series, threshold=3):
    return ((series - series.mean()) / series.std()).abs() > threshold

for col in ['data_gb', 'arpu_kes']:
    lo, hi = iqr_bounds(df[col])
    n_iqr = ((df[col] < lo) | (df[col] > hi)).sum()
    n_z   = zscore_outliers(df[col]).sum()
    print(f'{col:12s}  IQR outliers: {n_iqr:3d}  Z-score outliers: {n_z:3d}  '
          f'bounds=[{lo:.1f}, {hi:.1f}]')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(5)
N = 600
arpu = np.concatenate([np.random.lognormal(6.5,0.7,N), [5,50000,3,48000]])
df = pd.DataFrame({'arpu_kes': arpu})

q1, q3 = df['arpu_kes'].quantile(0.25), df['arpu_kes'].quantile(0.75)
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr

strategies = {
    'Drop outliers':    df[df['arpu_kes'].between(lo, hi)]['arpu_kes'],
    'Cap (Winsorise)':  df['arpu_kes'].clip(lo, hi),
    'Log transform':    np.log1p(df['arpu_kes']),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, vals) in zip(axes, strategies.items()):
    ax.hist(vals, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    ax.set_title(f'{name}\n(n={len(vals)})', fontweight='bold')
    ax.set_xlabel('ARPU (KES)')
plt.tight_layout()
fig.savefig('/tmp/day06_outliers.png', dpi=120)
print('Saved /tmp/day06_outliers.png')
plt.close()

## Outlier Handling Strategies

| Strategy | When to Use |
|----------|-------------|
| **Drop** | Clear data errors (e.g., age=200) |
| **Cap / Winsorise** | Keep row but limit extreme values |
| **Log/Power transform** | Reduce skewness while keeping all data |
| **Keep** | Valid extreme events (fraud amounts, viral tweets) |
| **Separate model** | Model extreme cases differently |

## 💪 Exercises

### Exercise 1
For the Safaricom dataset, find all duplicate `subscriber_id` rows. Keep only the row with the highest `arpu_kes` per duplicate group.

### Exercise 2
Apply IQR-based capping to the `data_gb` column. Plot before vs after box plots.

In [ ]:
# Your code here


## 📋 Summary

- ✓ `df.duplicated()` and `df.drop_duplicates()` handle duplicate rows
- ✓ IQR method: outlier if < Q1-1.5·IQR or > Q3+1.5·IQR
- ✓ Z-score method: outlier if |z| > 3
- ✓ Winsorisation (`clip`) keeps all rows while limiting extreme influence

## 🚀 Next Steps
**Day 7: Week 1 Review and Missing Data Mini-Project**